# ⚙️ Day 152 — CI/CD with GitHub Actions
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 152 (Month 8, Week 4, Day 2) |
| **Topic** | CI/CD · GitHub Actions · Automated Test → Build → Deploy Pipeline |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverables** | `ci.yml` workflow · `test_data.py` · `test_api.py` · `deploy.yml` · README badge |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 📊 Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals + ML Serving | 80/80 + 10★ ✅ |
| 148 | FastAPI + DB + Auth | 80/80 + 10★ ✅ |
| 149 | Streamlit ↔ FastAPI Full-Stack | 80/80 + 10★ ✅ |
| 150 | Docker + Containerization | 80/80 + 10★ ✅ |
| 151 | Cloud Deployment (SCC + Render) | 80/80 + 10★ ✅ |
| **152** | **CI/CD — GitHub Actions** | **← Today** |

**Running Total: 880/880 + 110★ 🔥 PERFECT**

---

## 🎯 Why This Day Matters

Day 151 taught you to deploy manually. Today you **automate** every deployment forever.

> **Freelance reality:** When you hand over a live dashboard to a client, they will ask:
> *"What happens when you push a bug fix? Is the site down?"*
> CI/CD is your answer: every push is tested before it touches production.
> This single skill separates ₹5k projects from ₹25k+ ongoing retainer contracts.

**What CI/CD solves:**

| Without CI/CD | With CI/CD |
|---|---|
| Push → hope it works | Push → tests run automatically |
| Manual `docker build` every time | Pipeline builds + pushes Docker image |
| Deploy by hand from your laptop | Render gets a deploy hook trigger |
| Broken code reaches production | Broken code is blocked at test stage |
| Client sees downtime | Client sees green badge on your README |


---
## 📦 Section 1 — Raw Data (Do Not Modify)

In [ ]:
# SECTION 1: RAW DATA — FreelanceHub India | seed=141 | 500 rows
# ══════════════════════════════════════════════════════════════
# This dataset is FROZEN. Never modify values in this section.
# All analysis happens in Sections 3 and 4.
# Column generation order is fixed to preserve numpy random state.
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

np.random.seed(141)
n = 500

platforms        = np.random.choice(["Upwork","Fiverr","Freelancer","Toptal","PeoplePerHour"], n)
categories       = np.random.choice(["Data Science","Web Dev","Design","Writing","Marketing"], n)
experience_level = np.random.choice(["Junior","Mid","Senior"], n)
hours_worked     = np.random.randint(5, 160, n)
hourly_rate      = np.round(np.random.uniform(500, 5000, n), 2)
client_rating    = np.round(np.random.uniform(3.0, 5.0, n), 1)
revenue          = np.round(hours_worked * hourly_rate * np.random.uniform(0.8, 1.2, n), 2)

RAW = pd.DataFrame({
    "platform"        : platforms,
    "category"        : categories,
    "experience_level": experience_level,
    "hours_worked"    : hours_worked,
    "hourly_rate"     : hourly_rate,
    "client_rating"   : client_rating,
    "revenue"         : revenue
})

print(f"Shape  : {RAW.shape}")
print(f"Columns: {list(RAW.columns)}")
print(f"Nulls  : {RAW.isnull().sum().sum()}")
print()
print(RAW.head())


---
## 📚 Section 2 — Concept Notes

### 2.1 What is CI/CD?

**CI (Continuous Integration):** Every push to your repo automatically runs tests, linting, and build checks. Catches bugs before they merge.

**CD (Continuous Delivery/Deployment):** After CI passes, the app is automatically deployed to staging or production. No manual steps.

```
You push code
    ↓
GitHub Actions triggers
    ↓
Job 1: test   → pytest + data validation
    ↓ (must pass)
Job 2: lint   → flake8 / black check
    ↓ (must pass)
Job 3: build  → docker build + push to Docker Hub
    ↓
Job 4: deploy → curl Render.com deploy hook
    ↓
Live app updated ✅
```

---

### 2.2 GitHub Actions Anatomy

```yaml
name: CI/CD Pipeline           # Display name in Actions tab

on:                            # TRIGGER: when does this run?
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:                          # JOBS: independent units of work
  test:                        # Job name (arbitrary)
    runs-on: ubuntu-latest     # Runner environment
    steps:                     # Sequential steps within the job
      - uses: actions/checkout@v4          # Clone your repo
      - uses: actions/setup-python@v5      # Install Python
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/ -v
```

---

### 2.3 Key Concepts

| Concept | What It Is | Example |
|---|---|---|
| **Trigger** | Event that starts a workflow | `push`, `pull_request`, `schedule` |
| **Job** | A group of steps on one runner | `test`, `build`, `deploy` |
| **Step** | A single command or Action | `run: pytest` |
| **Action** | Reusable step from Marketplace | `actions/checkout@v4` |
| **Runner** | VM that executes jobs | `ubuntu-latest` |
| **Secret** | Encrypted variable | `${{ secrets.RENDER_API_KEY }}` |
| **needs** | Job dependency | `deploy: needs: [test, build]` |
| **Artifact** | File passed between jobs | trained model, test report |

---

### 2.4 Job Dependencies — The Critical Concept

```yaml
jobs:
  test:
    runs-on: ubuntu-latest
    # ... runs independently

  build:
    runs-on: ubuntu-latest
    needs: test              # ← ONLY runs if test passes
    # ... if test fails, build is SKIPPED

  deploy:
    runs-on: ubuntu-latest
    needs: [test, build]     # ← ONLY runs if BOTH pass
    # ... broken code never reaches Render
```

**Why this matters for clients:** You can truthfully say *"nothing reaches production without passing all tests."* That's a professional guarantee.

---

### 2.5 Secrets Management in GitHub Actions

**Where to add secrets:** GitHub repo → Settings → Secrets and variables → Actions → New repository secret

```yaml
# Access in workflow:
env:
  RENDER_DEPLOY_HOOK: ${{ secrets.RENDER_DEPLOY_HOOK }}
  DOCKER_HUB_TOKEN: ${{ secrets.DOCKER_HUB_TOKEN }}

# Direct in step:
- name: Trigger Render deploy
  run: curl -X POST "${{ secrets.RENDER_DEPLOY_HOOK }}"
```

**Rule:** Never hardcode API keys in workflow files. They are public in your repo.

---

### 2.6 Writing Testable Code — pytest Basics for CI

```python
# tests/test_data.py
import pandas as pd
import numpy as np
import pytest

def get_dataset():
    np.random.seed(141)
    n = 500
    # ... same generation logic
    return df

def test_shape():
    df = get_dataset()
    assert df.shape == (500, 7)

def test_no_nulls():
    df = get_dataset()
    assert df.isnull().sum().sum() == 0

def test_revenue_positive():
    df = get_dataset()
    assert (df["revenue"] > 0).all()

def test_rating_range():
    df = get_dataset()
    assert df["client_rating"].between(3.0, 5.0).all()
```

Run locally: `pytest tests/ -v`
GitHub Actions runs the exact same command on every push.

---

### 2.7 README Status Badge

```markdown
![CI](https://github.com/deepanshu0110/Month8-Streamlit-FastAPI-Portfolio/actions/workflows/ci.yml/badge.svg)
```

This renders a live green ✅ / red ❌ badge on your GitHub README.
**Portfolio value:** Clients and employers see at a glance that your project has passing tests.

---

### 2.8 Workflow File Locations

```
your-repo/
├── .github/
│   └── workflows/
│       ├── ci.yml          ← test + lint on every push/PR
│       └── deploy.yml      ← build + deploy on push to main only
├── tests/
│   ├── test_data.py
│   └── test_api.py
├── fastapi_app/
├── streamlit_app/
└── ...
```

`.github/workflows/` is the **only** directory GitHub Actions reads. Filename = workflow name.

---

### 2.9 Caching Dependencies (Speed Optimization)

```yaml
- uses: actions/cache@v4
  with:
    path: ~/.cache/pip
    key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
    restore-keys: |
      ${{ runner.os }}-pip-

- name: Install dependencies
  run: pip install -r requirements.txt
```

**Effect:** First run: ~90s to install packages. Subsequent runs: ~5s (cache hit).
**Key insight:** `hashFiles('**/requirements.txt')` — cache invalidates automatically when requirements change.

---

### 2.10 Interview Answer

> *"How do you ensure code quality in your freelance projects?"*

**Model answer:**
*"I use GitHub Actions to run pytest and flake8 on every push. If tests fail, the Docker build and Render deployment are blocked by job dependencies. Clients get a README badge showing live pipeline status — they can see at any time whether the codebase is healthy without asking me."*


---
## 🛠️ Section 3 — Practice Tasks

Complete all tasks in order. Use only skills covered through Day 152.
Write your answers in the cells below each task prompt.


### Task 1 — Write the Workflow Trigger Block (10 pts)

Write a YAML trigger block (the `on:` section) that fires:
- On push to `main`
- On pull request to `main`
- Every Monday at 6:00 AM UTC (scheduled)

Store the YAML as a Python multi-line string called `trigger_yaml`.
Print it. Then answer: what GitHub Actions keyword separates individual schedule entries?


In [ ]:
# ── Task 1: YOUR ANSWER ──────────────────────────────────────
# Write trigger_yaml as a string.
# Answer the keyword question in a print statement.

# trigger_yaml = """
# on:
#   ...
# """

# print(trigger_yaml)
# print("Keyword that separates schedule entries: ...")


### Task 2 — Write the Data Validation Test File (15 pts)

Write the complete content of `tests/test_data.py` as a Python string.

It must contain **5 pytest test functions** that check:
1. Dataset shape is (500, 7)
2. Zero null values
3. All `revenue` values are positive
4. `client_rating` is between 3.0 and 5.0 (inclusive)
5. `platform` column contains exactly 5 unique values

Store as `test_data_content` string and print it.
Then print what command you'd run locally to execute these tests with verbose output.


In [ ]:
# ── Task 2: YOUR ANSWER ──────────────────────────────────────
# Write test_data_content as a multi-line string.
# Include all 5 test functions with proper assert statements.

# test_data_content = """
# import ...
# """

# print(test_data_content)
# print("Local run command: ...")


### Task 3 — Write the Job Dependency Structure (10 pts)

Write the `jobs:` section of a workflow with 4 jobs:
- `test` — runs independently
- `lint` — needs `test`
- `build-docker` — needs `test`  
- `deploy` — needs **both** `lint` and `build-docker`

Store as `jobs_yaml` string. Print it.
Then answer: if `test` fails, which jobs are skipped? List all of them.


In [ ]:
# ── Task 3: YOUR ANSWER ──────────────────────────────────────
# Write jobs_yaml with the 4 jobs and their needs: fields.

# jobs_yaml = """
# jobs:
#   ...
# """

# print(jobs_yaml)
# print("Jobs skipped if test fails: ...")


### Task 4 — Write the Secrets Reference Block (10 pts)

Write the `env:` block for a deploy job that exposes three secrets:
- `RENDER_DEPLOY_HOOK` → from GitHub secret `RENDER_DEPLOY_HOOK`
- `DOCKER_HUB_USERNAME` → from GitHub secret `DOCKER_HUB_USERNAME`
- `DOCKER_HUB_TOKEN` → from GitHub secret `DOCKER_HUB_TOKEN`

Also write the single `run:` step that triggers the Render deploy using a `curl` POST request.

Store both as strings. Print them.
Then answer: why must secrets never be hardcoded in workflow files?


In [ ]:
# ── Task 4: YOUR ANSWER ──────────────────────────────────────
# env_block = """
# env:
#   ...
# """

# deploy_step = """
# - name: Trigger Render deploy
#   run: ...
# """

# print(env_block)
# print(deploy_step)
# print("Why no hardcoding: ...")


### Task 5 — Write the Dependency Cache Step (10 pts)

Write the complete `steps:` block for a job that:
1. Checks out the repo (`actions/checkout@v4`)
2. Caches pip dependencies using `actions/cache@v4`
   - Cache path: `~/.cache/pip`
   - Key: based on OS and hash of `requirements.txt`
3. Installs from `requirements.txt`
4. Runs `pytest tests/ -v --tb=short`

Store as `steps_yaml` string and print it.
Then answer: what happens to the cache when `requirements.txt` changes?


In [ ]:
# ── Task 5: YOUR ANSWER ──────────────────────────────────────
# steps_yaml = """
# steps:
#   - uses: actions/checkout@v4
#   ...
# """

# print(steps_yaml)
# print("Cache behavior when requirements.txt changes: ...")


### Task 6 — Write the README Badge + NRA (15 pts)

**Part A (5 pts):** Write the complete Markdown badge string for the `ci.yml` workflow
in the `deepanshu0110/Month8-Streamlit-FastAPI-Portfolio` repo.
Store as `badge_md` and print it.

**Part B (10 pts):** Write an NRA analysis for CI/CD business value.

Use this structure:
```
Number  : [specific metric or count]
Reason  : [why this number matters technically]
Action  : [specific next step — name the file, state the threshold]
```

The NRA must reference: deployment frequency, test gate, and pipeline file location.


In [ ]:
# ── Task 6: YOUR ANSWER ──────────────────────────────────────
# Part A
# badge_md = "![CI](...)"
# print(badge_md)

# Part B
# print("Number  :", ...)
# print("Reason  :", ...)
# print("Action  :", ...)


### Task 7 — Assemble the Complete ci.yml (10 pts)

Assemble a complete, deployable `.github/workflows/ci.yml` file as a single string.

It must include:
- `name: CI Pipeline`
- Trigger: push + pull_request on `main`
- One job called `test` with:
  - `runs-on: ubuntu-latest`
  - Python 3.11 setup
  - pip cache step
  - Install from `requirements.txt`
  - Run `pytest tests/ -v`
- One job called `deploy` that:
  - `needs: test`
  - Runs only on push (not PR): `if: github.event_name == 'push'`
  - Triggers Render deploy via curl using `secrets.RENDER_DEPLOY_HOOK`

Store as `ci_yml_content` string. Print it.


In [ ]:
# ── Task 7: YOUR ANSWER ──────────────────────────────────────
# ci_yml_content = """
# name: CI Pipeline
# on:
#   ...
# jobs:
#   test:
#     ...
#   deploy:
#     needs: test
#     if: github.event_name == 'push'
#     ...
# """

# print(ci_yml_content)


---
## 🏆 Section 4 — Scoring Rubric

| Task | Max Points | What's Graded |
|---|---|---|
| T1 — Trigger Block | 10 | All 3 trigger types present; cron syntax correct (`0 6 * * 1`); keyword question answered |
| T2 — test_data.py | 15 | All 5 test functions present; correct assert logic; pytest run command correct |
| T3 — Job Dependencies | 10 | 4 jobs present; `needs:` values correct; skipped jobs listed correctly (all 3) |
| T4 — Secrets Block | 10 | All 3 secrets referenced with `${{ secrets.X }}` syntax; curl deploy step correct; security reason explained |
| T5 — Cache Step | 10 | `actions/cache@v4` used; `hashFiles` key correct; all 5 steps present; invalidation explained |
| T6 — Badge + NRA | 15 | Badge URL exact match; NRA has specific number (0 steps), specific reason (test gate), specific action (filename + secret name) |
| T7 — Full ci.yml | 10 | Complete valid YAML; both jobs present; `if: github.event_name == 'push'` on deploy; cache + needs correct |
| **Total** | **80** | |
| ⭐ Bonus — deploy.yml | 10★ | Write a separate `deploy.yml` that also builds + pushes to Docker Hub on push to main |

---

### NRA Grading Criteria (Tasks 6B)

| Element | Weak (−5) | Strong (+10) |
|---|---|---|
| **Number** | "faster deployments" | "0 manual deployment steps after ci.yml commit" |
| **Reason** | "CI/CD is good practice" | "failing test blocks curl via needs: dependency — broken code cannot reach Render" |
| **Action** | "set up CI/CD" | "commit .github/workflows/ci.yml; add RENDER_DEPLOY_HOOK to GitHub Secrets" |

---

### Interview Framing

> **Q: "What does your development workflow look like on a freelance project?"**

**Strong answer:**
*"I commit a `.github/workflows/ci.yml` to every project. On each push to main, GitHub Actions runs pytest against my data validation suite — if any test fails, the Render deploy hook never fires. Clients see a live badge on the README. It means I can push a fix at midnight and the client wakes up to an updated app, not a broken one. Setup time is about 20 minutes per project."*

**What makes this strong:**
- Names the specific tool (GitHub Actions) and file (`.github/workflows/ci.yml`)
- States the client-visible benefit (live badge, no breakage)
- Quantifies setup cost (20 minutes) — shows it's a habit, not a one-off
